```plaintext
.
└── 📂 output
    ├── 📂 styles                                     # Результаты работы обучающих скриптов
    │   └── 📂 *experiment_name*
    │       ├── 📂 images
    │       │   ├── 🖼️ comparison_*step*.png          # Сравнение сеток изображений Source-Target
    │       │   ├── 📷 source.png                     # Сетка Source
    │       │   └── 🎨 target_*step*.png              # Сетка Target
    │       ├── 📂 weights
    │       │   ├── 🧠 *experiment_name*_*step*.pt    # Веса модели каждые save_weights_every_n шагов
    │       │   └── 🧠 *experiment_name*.pt           # Веса модели на последнем шаге num_steps
    │       └── ⚙️ config.json                        # Параметры, с которыми была вызвана функция train()
    └── 📂 inversion_bank                             # Результаты работы скриптов инверсии
        └── 📂 *person_name*
            ├── 🖼️ image_*step*.png                   # Результат инверсии на шаге step
            ├── 🖼️ image.png                          # Результат инверсии на последнем шаге
            ├── 💾 w_*step*.pt                        # Инвертированный латент на шаге step
            └── 💾 w.pt                               # Инвертированный латент на последнем шаге
```

# Imports

In [ ]:
import os
import gc
import torch
import warnings

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Literal
from collections.abc import Collection

from src.train import train
from src.utils.fix_random import seed_everything
from src.utils.weights import load_base_generator, load_styled_generator
from src.utils.image import generate_one_image, tensor_to_pil
from src.pipelines.latent_optimization import LatentOptimizationPipeline
from src.pipelines.e4e_inversion import E4EInversionPipeline

# Сменим директорию на корень проекта, если ноутбук запущен не из корня, чтобы папки
# weights, output и т.д. создавались корректно, в корне, а не в notebooks:
current_dir = Path(os.getcwd())
if current_dir.name == "notebooks":
    os.chdir(current_dir.parent)

# Внутри библиотеки lpips используется старый способ загрузки весов, уберем предупреждения:
warnings.filterwarnings('ignore', category=UserWarning, module='torchvision.models._utils')

# Training

## Quick Guide

**ПАРАМЕТРЫ ОБУЧЕНИЯ**

---
**Основные:**

- `source_text` / `target_text` — исходный и целевой промты, задающие направление смены домена.
- `experiment_name` — краткое название текущего запуска. В папке `output/styles` будет создана директория с таким названием с результатами работы скрипта.
- `resolution` — выходное разрешение картинки. Возможные значения: 256, 512, 1024. Скрипт загрузит модель StyleGAN2 и веса для заданного разрешения автоматически.
- `device` — устройство, на котором будет осуществляться обучение.
- `seed` — сид, с которым будет осуществляться обучение. Нужен для обеспечения повторяемости результатов экспериментов. Если передан `None`, сид не фиксируется.
---
**Гиперпараметры NADA:**

- `num_steps` — количество шагов обучения.
- `batch_size` — размер батча.
- `lr` — шаг обучения.
- `truncation_psi` — коэффициент усечения распределения (латентного пространства $\mathcal{Z}$). Используется при обучении NADA, отборе блоков для заморозки и при валидации.
---
**CLIP:**

- `clip_model_name` — название модели CLIP из списка: RN50, RN101, RN50x4, RN50x16, RN50x64, ViT-B/32, ViT-B/16, ViT-L/14, ViT-L/14@336px. Используется для CLIP-based лоссов, будет загружена скриптом автоматически.
---
**Параметры выбора блоков для заморозки:**

- `blocks_selection_mode` — режим выбора блоков для заморозки из списка: `static`, `once`, `adaptive`.
    + `static` — выбор блоков в процессе обучения не производится, замораживаются явно переданные `blocks_to_freeze` блоки.
    + `once` — выбор блоков производится один раз до цикла обучения и не меняется в процессе.
    + `adaptive` — выбор блоков осуществляется динамически в процессе обучения каждые `adaptive_selection_every_n` шагов.
- `blocks_to_freeze` — кортеж или список блоков, которые должны быть заморожены в процессе обучения. Параметр учитывается, если выбран режим `static`.

    > StyleGAN2 содержит следующие блоки:
    > * для разрешения 256: ('b4', 'b8', 'b16', 'b32', 'b64', 'b128', 'b256')
    > * для разрешения 512: ('b4', 'b8', 'b16', 'b32', 'b64', 'b128', 'b256', 'b512')
    > * для разрешения 1024: ('b4', 'b8', 'b16', 'b32', 'b64', 'b128', 'b256', 'b512', 'b1024')

- `k_trainable_blocks` — количество блоков, которые будут выбраны для обучения (останутся размороженными). StyleGAN2 всего содержит 7 блоков для разрешения 256, 8 блоков для 512 и 9 блоков для 1024. Параметр учитывается, если выбран режим `once` или `adaptive`.
- `select_batch_size` — размер батча при отборе обучаемых блоков. Global CLIP Loss будет считаться и усредняться по такому количеству случайных латентов. Чем выше, тем менее стохастичным будет процесс отбора блоков. Если лосс считается по одному изображению, полученному от случайного латента с краев распределения, то результат может не отражать нужное направление для всего распределения, а если усредняется по нескольким изображениям, то результат становится стабильнее и предсказуемее. Параметр учитывается, если выбран режим `once` или `adaptive`.
- `select_num_steps` — количество шагов оптимизации тензора латентов $\mathcal{w}$ в пространстве $\mathcal{W+}$ при отборе обучаемых блоков. При большем значении латент оптимизируется сильнее, оптимизатор успевает накопить моменты для глубоких слоев. Параметр учитывается, если выбран режим `once` или `adaptive`.
- `select_lr` — шаг обучения при отборе обучаемых блоков. Параметр учитывается, если выбран режим `once` или `adaptive`.
- `select_criterion` — критерий отбора блоков: `absolute` или `relative`. Параметр учитывается, если выбран режим `once` или `adaptive`.

    > $\Delta \mathbf{w} = \mathbf{w_{opt}} - \mathbf{w_{base}}$
    >
    > **`absolute`** — среднее абсолютное отклонение. Оценивает абсолютный сдвиг весов вне зависимости от их исходного масштаба. Выделяет блоки, слои которых физически меняются сильнее всего.
    > $$\left\langle \|\Delta \mathbf{w}\| \right\rangle$$
    > **`relative`** — среднее относительное отклонение. Делит норму изменения каждого блока на его собственную базовую норму. То есть приводит к масштабу исходного вектора стиля этого конкретного блока. Выделяет блоки с наибольшими относительными изменениями, нивелируя разницу масштабов на разных разрешениях сети.
    > $$\left\langle \frac{\|\Delta \mathbf{w}\|}{\|\mathbf{w}_{\text{base}}\|} \right\rangle$$

- `select_norm` — тип нормы при отборе обучаемых блоков из списка: `l1`, `l2`. Параметр учитывается, если выбран режим `once` или `adaptive`.

    > **`l1`** — сумма абсолютных значений координат. Дает одинаковый вес как большим, так и маленьким изменениям, важна лишь общая сумма абсолютных значений всех координат латента. Может работать лучше, когда стиль предполагает небольшие комплексные сдвиги по многим координатам латента, то есть небольшие изменения по всей фичемапе. Например, какая-то общая для изображения текстура.
    >
    > **`l2`** — корень из суммы квадратов координат. За счет возведения в квадрат сильнее штрафует небольшие изменения, а вес больших по модулю координат латента, напротив, увеличивается. Может работать лучше, когда для стиля важны блоки, в которых есть большие локальные изменения (хотя бы несколько координат латента изменились очень сильно). Например, сильное локальное изменение геометрии или яркости. 

- `adaptive_selection_every_n` — определяет, с какой периодиодичностью осуществляется отбор обучаемых блоков во время обучающего цикла. Параметр учитывается, если выбран режим `adaptive`.

**Примечание:**

- Комбинация параметров `select_criterion` и `select_norm` может существенно влиять на результат. Например, из исходного улыбающегося человека может получиться как свирепый оборотень, так и улыбающийся человек с чертами оборотня.
- При помощи комбинации параметров `select_num_steps` и `adaptive_selection_every_n` можно реализовать концептуально разные подходы к обучению.
    1. `select_num_steps` = 1, `adaptive_selection_every_n` = 1 — полностью адаптивное обучение, на каждом шаге производится отбор слоев, но этот отбор происходит "поверхностно" (с малым числом шагов оптимизации, например, 1), чтобы обучение длилось не слишком долго.
    2. `select_num_steps` = 200, `adaptive_selection_every_n` = 50 — интервальный подход. Идея в том, что мы проводим тщательный отбор наиболее важных на текущий момент блоков, после чего обучаем сеть на протяжении какого-то количества шагов, чтобы отобранные слои успели достаточно обучиться, после чего снова выбираем обучаемые слои.
    3. Градации между этими вариантами. 

---
**Параметры директорий, логов, валидации, сохранения чекпоинтов:**

- `weights_dir` — директория с предобученными весами используемых моделей.
- `output_dir` — директория, в которую сохраняются результаты работы скрипта (валидационные сетки изображений, веса обученного генератора, конфиг с параметрами обучения).
- `init_generator_weights_path` — путь к весам, которыми будет инициализирован генератор. Параметр нужен для реализации Fine-Tuning. Вы можете передать веса уже стилизованного генератора и обучить его по другому промпту. По умолчанию `None`, в этом случае инициализируется базовый генератор.
- `save_changed_only` — флаг, сохранять ли, если возможно, только измененные веса генератора (с `requires_grad = True`) или всю сеть `generator.synthesis`. По умолчанию, если позволяет режим обучения, сохраняются только измененные веса. Функция `load_styled_generator` поддерживает загрузку частично сохраненных весов. Если этот параметр передается в консоль, то вместо `--save_changed_only False` следует указывать `--no-save_changed_only`.
- `logging_every_n` — частота логирования в консоль. Параметр учитывается, если `verbose` > 0.
- `save_weights_every_n` — частота сохранения промежуточных весов генератора. Если передан `None`, сохраняются только конечные веса на шаге `num_steps`.
- `validate_every_n` — частота валидации (сохранения сеток валидационных изображений). Удобно задавать равной `save_weights_every_n`, чтобы были веса генератора для каждого этапа переноса стиля.
- `val_set_path` — путь к заранее сохраненным латентам, по которым осуществляется валидация. Файл с латентами уже имеется в репозитории, но если очень хочется сгенерировать свои валидационные латенты, например, сделать сетку изображений больше, то из папки *data* нужно удалить *fixed_val_set.pt*, после чего сгенерировать новый:
    ```python
    from src.utils.validation import create_fixed_validation_set

    create_fixed_validation_set(num_samples=*your_num*, seed=*your_seed*)
    ```
- `verbose` — подробность выводов в консоль, возможные значения: 0, 1, 2.
    + 0 — минимальное количество выводов, только самые важные оповещения.
    + 1 — оповещения и базовые логи (номер шага, значение лосса) каждые `logging_every_n` шагов.
    + 2 — оповещения, базовые логи, а также список замороженных в данный момент слоев, если выбран адаптивный режим обучения, каждые `logging_every_n` шагов.

## BIG RED BUTTON

In [ ]:

@dataclass
class TrainConfig:
    # Основное
    source_text: str = 'a photo of a person'
    target_text: str = 'a sketch of a person'
    experiment_name: str = 'sketch'
    resolution: Literal[256, 512, 1024] = 1024
    device: torch.device | Literal['cuda', 'cpu'] | str = 'cuda'
    seed: int | None = 101
    
    # Гиперпараметры NADA
    num_steps: int = 300
    batch_size: int = 4
    lr: float = 0.002
    truncation_psi: float = 0.7
    
    # Какую модель использовать для CLIP-based лоссов
    clip_model_name: Literal[
        'RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64',
        'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px'
    ] = 'ViT-B/32'
    
    # Параметры выбора блоков для заморозки
    blocks_selection_mode: Literal['static', 'once', 'adaptive'] = 'static'
    blocks_to_freeze: Collection[str] = ('b4', 'b8', 'b16', 'b32')
    k_trainable_blocks: int = 3
    select_batch_size: int = 4
    select_num_steps: int = 50
    select_lr: float = 0.01
    select_criterion: Literal['absolute', 'relative'] = 'absolute'
    select_norm: Literal['l1', 'l2'] = 'l2'
    adaptive_selection_every_n: int = 50
    
    # Параметры логов, валидации, сохранения чекпоинтов
    weights_dir: Path | str = 'weights'
    output_dir: Path | str = 'output/styles'
    init_generator_weights_path: Path | str | None = None
    save_changed_only: bool = True
    logging_every_n: int = 10
    save_weights_every_n: int | None = 50
    validate_every_n: int | None = 50
    val_set_path: Path | str | None = Path('data/fixed_val_set.pt')
    verbose: Literal[0, 1, 2] = 2
    
cfg = TrainConfig()

seed_everything(cfg.seed)

train(**cfg.__dict__)

gc.collect()
torch.cuda.empty_cache()

# Inference

**ПАРАМЕТРЫ ИНФЕРЕНСА**

---
- `style_weights_path` — путь к весам обученного генератора. Обычно или **output/styles/EXPERIMENT_NAME/weights/FILENAME.pt** или **pretrained_weights/FILENAME.pt**.
- `resolution` — выходное разрешение картинки. Возможные значения: 256, 512, 1024.
- `weights_dir` — директория с предобученными весами используемых моделей.
- `device` — устройство, на котором будет осуществляться инференс.

In [ ]:
@dataclass
class InferenceConfig:
    style_weights_path: Path | str = Path('output/styles/sketch/weights/sketch.pt')
    resolution: Literal[256, 512, 1024] = 1024
    weights_dir: Path | str = 'weights'
    device: torch.device | Literal['cuda', 'cpu'] | str = 'cuda'
    
cfg = InferenceConfig()

generator = load_styled_generator(**cfg.__dict__)

In [ ]:
img = generate_one_image(generator, truncation_psi=0.7)
display(img)

# Invertion

## Quick Guide

В настоящий момент реализовано два способа инверсии реального изображения — инверсия с помощью `e4e` и латентная оптимизация в $\mathcal{W+}$ пространстве. К сожалению, для `e4e` нет официальных предобученных весов для разрешений 256 и 512. Возможно, позже я дообучу `e4e` под меньшие разрешения, но пока что его можно применить только для разрешения 1024. Если ваш генератор обучен в разрешении 256 или 512, используйте пайплайн латентной оптимизации.

Разница в этих двух подходах следующая:

- `e4e` — это предобученная сеть-энкодер, которая обучалась восстанавливать латентный код по изображению, оставаясь при этом в тех областях латентного пространства, которые хорошо поддаются редактированию. Процесс инверсии этим способом — это просто инференс сети. Инверсия происходит быстро, но не всегда хорошо передает узнаваемые черты лица, потому что сеть намеренно обучалась работать ближе к центру распределения.
- Латентная оптимизация — это итеративный процесс оптимизации тензора латентов $\mathcal{w}$ в пространстве $\mathcal{W+}$ градиентным спуском при помощи связки лоссов: MAE, MSE, LPIPS, ID-Loss (ArcFace), Reg-Loss (MSE между текущим и средним латентами) с соответствующими коэффициентами. При тщательном подборе гиперпараметров может давать довольно точный и узнаваемый результат.

Если вы работаете с разрешением 1024, то вам доступен гибридный подход — вы можете использовать латент, полученный с помощью `e4e` в качестве стартовой точки для латентной оптимизации.

**ПАРАМЕТРЫ ИНВЕРСИИ**

---
**Основные:**

- `style_weights_path` — путь к весам обученного генератора. Обычно или **output/styles/EXPERIMENT_NAME/weights/FILENAME.pt** или **pretrained_weights/FILENAME.pt**.
- `resolution` — выходное разрешение картинки. Возможные значения: 256, 512, 1024.
- `weights_dir` — директория с предобученными весами используемых моделей.
- `device` — устройство, на котором будет осуществляться инверсия.
---
**Гиперпараметры латентной оптимизации:**

- `lpips_net` — какую предобученную сеть использовать для вычисления LPIPS-лосса. Возможные значения: 'alex', 'vgg'.
- `lr` — шаг обучения латентной оптимизации.
- `gamma` — коэффициент затухания шага обучения в `lr_scheduler.ExponentialLR`.
- `lambda_l1`, `lambda_l2`, `lambda_lpips`, `lambda_id`, `lambda_reg` — коэффициенты при соответствующих лоссах.

    > `L1 (MAE)` и `L2 (MSE)` лоссы отвечают за попиксельное сходство исходного и восстановленного изображения.
    >
    > `LPIPS` лосс отвечает за визуальное восприятие сходства изображений. Пропускает исходное и восстановленное изображения через сверточную сеть и сравнивает карты признаков, отвечающие за общую геометрию, формы и текстуры изображения.
    >
    > `ID-Loss` отвечает за соответствие черт лица и узнаваемость конкретного человека.
    >
    > `Reg-Loss` — это `MSE` между `w_opt` и `w_avg`. То есть, это регуляризация, которая заставляет модель искать хорошее решение вблизи к центру латентного распределения `w_avg`. Нужен для сохранения редактируемости изображения при помощи StyleGAN2.
---
**Параметры директорий, логов и сохранения чекпоинтов латентной оптимизации:**

- `w_avg_dir` — директория с файлами средних латентов `w_avg`. Эти тензоры заранее вычислены и сохранены в `data/w_avg_*RESOLUTION*.pt`. Вычисляются запуском скрипта `src/utils/invertion.py`.
- `output_dir` — директория, в которую сохраняются результаты латентной оптимизации (восстановленные изображения и соответствующие им латенты).
- `save_every_n` — частота сохранения восстановленных изображений и соответствующих им латентов. Если передан `None`, сохраняются только конечные изображения и латенты на шаге `steps`.
- `logging_every_n` — частота логирования в консоль. Параметр учитывается, если `verbose` > 0.
- `verbose` — подробность выводов в консоль, возможные значения: 0, 1.
    + 0 — минимальное количество выводов, только самые важные оповещения.
    + 1 — оповещения и базовые логи (номер шага, значения лоссов) каждые `logging_every_n` шагов.
---
**Параметры вызова метода `__call__`:**

- `image_path` — путь к инвертируемому изображению.
- `person_name` — имя человека, чья фотография инвертируется. Будет создана директория `output/inversion_bank/person_name`, в которую будут сохранены результаты латентной оптимизации.
- `steps` — число шагов оптимизации.
- `w_init` — тензор, которым будет инициализирован оптимизируемый латент `w_opt`. Если передан `None`, то `w_opt` инициализируется средним латентом `w_avg`. Сюда можно передать латент, полученный с помощью `e4e`. 
- `initial_noisy_steps` — сколько первых шагов к `w_opt` будет добавляться небольшой шум. Иногда помогает выбраться из локальных минимумов в центре распределения и ускорить сходимость при инициализации через `w_avg`. Если передано 0 шагов, то шум не добавляется. При инициализации через `w_init` через `e4e` лучше оставлять равным 0.
- `noise_alpha` — максимальная амплитуда случайного шума. Амплитуда линейно уменьшается до нуля за `initial_noisy_steps` шагов.

## Processing

In [ ]:
@dataclass
class InvertionConfig:
    style_weights_path: Path | str = Path('output/styles/sketch/weights/sketch.pt')
    resolution: int = 1024
    weights_dir: Path | str = 'weights'
    device: torch.device | Literal['cuda', 'cpu'] | str = 'cuda'
    
    # Гиперпараметры латентной оптимизации
    lpips_net: Literal['alex', 'vgg'] = 'vgg'
    lr: float = 0.1
    gamma: float = 0.99
    lambda_l1: float = 1.0
    lambda_l2: float = 0.1
    lambda_lpips: float = 0.8
    lambda_id: float = 0.5
    lambda_reg: float = 0.01
    
    # Параметры директорий, логов и сохранения чекпоинтов латентной оптимизации
    w_avg_dir: Path | str = 'data'
    output_dir: Path | str = 'output/inversion_bank'
    save_every_n: int | None = 50
    logging_every_n: int = 10
    verbose: Literal[0, 1] = 1

cfg = InvertionConfig()

opt_params = asdict(cfg)
opt_params.pop('style_weights_path', None)

# Посмотреть, как сработала инверсия, можно на обычном или стилизованном генераторе:
generator = load_base_generator(cfg.resolution, cfg.weights_dir, cfg.device)
# generator = load_styled_generator(cfg.style_weights_path, cfg.resolution, cfg.weights_dir, cfg.device)

# Можно выбрать один из пайплайнов инверсии или оба сразу:
pipeline_e4e = E4EInversionPipeline(cfg.weights_dir, cfg.device)
pipeline_opt = LatentOptimizationPipeline(**opt_params)

In [ ]:
image_path = Path('data/will_smith.jpg')

w = pipeline_e4e(image_path)

# Вы также можете загрузить ранее сохраненный латент:
# w = torch.load('output/inversion_bank/PERSON_NAME/FILENAME.pt', map_location=cfg.device)

w = pipeline_opt(
    image_path=image_path,
    person_name='will_smith',
    steps=200,
    # w_init=None,  # Инициализация через w_avg
    w_init=w,       # Инициализация через e4e 
    initial_noisy_steps=0,
    noise_alpha=0.05
)

with torch.inference_mode():
    img_tensor = generator.synthesis(w)

img = tensor_to_pil(img_tensor)

# Можно сохранить латент и картинку, если нужно:
# torch.save(w.detach().cpu(), 'your/save/path.pt')
# img.save('your/save/path.png')

display(img)